# Redact CVs, augment survey data, and build the ORPO dataset

Cleaned archival copy of preparation cells 4-9 from `emnlp.ipynb`. Later training, Hub publication, prototype dataset conversion, and older evaluation experiments are excluded. No study records, saved outputs, CV text, or model weights are included.

Core narrative and contrast-generation functions, prompts, positional field slices, combinatorial construction, and cleanup behavior are retained. Paths, record displays, authentication, and external-use safeguards are release adaptations, not evidence that those safeguards existed during the historical study. See `docs/REPRODUCIBILITY.md`.

Inputs remain private: an authorized `responses.csv` and corresponding PDFs in a `CV` subdirectory. Output CSVs retain private survey/derived content and must not be published. Redaction is not a certificate of anonymity. Review every field intended for external processing and use only inputs authorized for that processing. The institution/company omission prompt constrains the answer, not the request.

Historical dependencies are recorded in `requirements/declared_preparation.txt`; they are not a tested installation lock. A compatible English Presidio NLP model is required separately. No model installation or download is triggered automatically by this notebook's setup cell.


In [ ]:
import os
import re
import time
import unicodedata
from itertools import product
from pathlib import Path

import fitz
import pandas as pd
from openai import OpenAI
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from tqdm import tqdm


def preparation_client():
    if os.environ.get("CAEE_ALLOW_EXTERNAL_PREPARATION") != "1":
        raise RuntimeError("External preparation is disabled. Explicit API opt-in is required.")
    if os.environ.get("CAEE_PREPARATION_INPUTS_REVIEWED") != "1":
        raise RuntimeError("Review and authorize all selected request inputs before enabling preparation.")
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY must be supplied securely through the environment.")
    return OpenAI(api_key=os.environ["OPENAI_API_KEY"])


def validate_master_schema(frame):
    expected = {"id": 0, "Full name": 101, "E-mail address": 102,
                "LinkedIn profile page": 103, "Network ID": 110, "cv_text": 112}
    if len(frame.columns) != 113 or any(frame.columns[index] != name for name, index in expected.items()):
        raise ValueError("The historical positional slices require the documented 113-column schema.")


input_location = os.environ.get("CAEE_PREPARATION_INPUT_DIR")
if not input_location:
    raise RuntimeError("CAEE_PREPARATION_INPUT_DIR must point to an authorized private input folder.")
input_dir = Path(input_location).expanduser().resolve()
survey_path = input_dir / "responses.csv"
if not survey_path.is_file():
    raise FileNotFoundError("The authorized input folder must contain responses.csv.")
output_dir = Path(os.environ.get("CAEE_PREPARATION_OUTPUT_DIR", "outputs/preparation")).expanduser().resolve()
output_dir.mkdir(parents=True, exist_ok=True)


## CV preprocessing (source cell 4)

Presidio and literal name-part replacement run before the narrative-generation section in the written pipeline. The master output still retains the original survey columns. Diagnostic prints of CV text, detected entities, and survey records have been removed. Keep this master file private.


In [ ]:
#Load the dataset from Google Drive
dataset_path = survey_path
responses = pd.read_csv(dataset_path)


pdf_dir = input_dir / "CV"

# Function to extract text from a PDF file
def extract_text_from_pdf(pdf_path):
    text = ""
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text += page.get_text()
    return text

# Function to cleanse and anonymize text
def anonymize(text):

    # Initialize the analyzer and anonymizer engines
    analyzer = AnalyzerEngine()
    anonymizer = AnonymizerEngine()

      # Remove all newline characters and join the text into a single string
    data_str = ''.join(str(item).replace("\n", " ") for item in text)

    # Analyze text to find PII entities
    results = analyzer.analyze(text=data_str, entities=["PHONE_NUMBER", "EMAIL_ADDRESS", "PERSON", "URL"], language='en')

    # Anonymize the text based on analysis results
    anonymized_result = anonymizer.anonymize(text=data_str, analyzer_results=results)
    anonymized_text = anonymized_result.text

    return anonymized_text

# Function to check and replace names in the text
def replace_names(text, full_name):
    # Split full name into first and last names
    name_parts = full_name.split()
    for part in name_parts:
        text = text.replace(part, "<NAME>")
    return text

# Add a new column to the dataframe for the PDF text
responses['cv_text'] = ""

missing_cv_count = 0

# Iterate through the rows of the dataframe and load corresponding PDF text
for index, row in responses.iterrows():
    pdf_filename = f"{row['id']}.pdf"
    pdf_path = os.path.join(pdf_dir, pdf_filename)

    if os.path.exists(pdf_path):
        pdf_text = extract_text_from_pdf(pdf_path)
        anonymized_text = anonymize(pdf_text)

        # Further anonymize by replacing full name and parts of the name
        full_name = row['Full name']
        fully_anonymized_text = replace_names(anonymized_text, full_name)

        responses.at[index, 'cv_text'] = fully_anonymized_text
    else:
        missing_cv_count += 1

# Display the updated dataframe

# Save the updated dataframe back to Google Drive
output_csv_path = output_dir / "full_responses.csv"
responses.to_csv(output_csv_path, index=False)
print(f"Saved private master with {len(responses)} rows; missing CVs: {missing_cv_count}.")


## Narrative generation (source cell 5)

Requires explicit external-use opt-in, reviewed/authorized inputs, and an API key. The schema guard protects the documented positional mapping, not privacy by itself. The six groups receive three variants each. The additional-information slice is retained as an unused source declaration. API calls may incur charges.


In [ ]:
dataset_path = output_dir / "full_responses.csv"
full_responses = pd.read_csv(dataset_path)
validate_master_schema(full_responses)
client = preparation_client()

# Extract columns 3 to 19 (included)
academic_columns = full_responses.iloc[:, 2:19]

# Extract columns 20 to 36 for professional experience
professional_columns = full_responses.iloc[:, 19:35]

# Extract columns 37 to 42 for innovation information
innovation_columns = full_responses.iloc[:, 35:41]

# Extract columns 37 to 42 for entrepreneurship information
entrepreneurship_columns = full_responses.iloc[:, 41:76]

# Extract columns 37 to 42 for entrepreneurship information
motivation_columns = full_responses.iloc[:, 76:100]

# Extract columns 37 to 42 for additional information
addinfo_columns = full_responses.iloc[:, 100:101]

# Extract columns 37 to 42 for CV information
cv_columns = full_responses.iloc[:, 112:113]

def generate_descriptions(row, headers, base_prompt, client):
    details = [f"{header}: {row[i]}" for i, header in enumerate(headers) if pd.notna(row[i])]

    prompts = [
        base_prompt + " Please narrate in a concise manner, without specifying any institutions or company names.",
        base_prompt + " Please provide a detailed narrative without specifying any institutions or company names.",
        base_prompt + " Please provide a concise narrative with any additional insights you think are relevant, without specifying any institutions or company names."
    ]

    responses = []

    for prompt in prompts:
      response = client.chat.completions.create(
        model="gpt-3.5-turbo-0125",
        messages=[
                {"role": "system", "content": "You are a helpful research assistant, and you are going to assist me reviewing survey data"},
                {"role": "user", "content": f"{prompt}: {', '.join(details)}"}
        ],
        max_tokens=500
        )
      responses.append(response.choices[0].message.content)

    return responses

# Initialize new columns for academic information
full_responses['academic_description_v1'] = ""
full_responses['academic_description_v2'] = ""
full_responses['academic_description_v3'] = ""

# Headers for academic columns
academic_headers = academic_columns.columns.tolist()

# Base prompt for academic descriptions
academic_prompt = """
List in chronological order the academic background of this professional, including degrees obtained and the years they were completed.
Identify the highest degree accomplished and the year it was awarded.
Mention the year when the professional first graduated from a higher education institution.
Describe any gap years between academic degrees and note if the professional changed institutions between degrees, providing context on whether these gaps might indicate periods of employment or other activities.
"""

# Iterate through each row and generate academic descriptions
for index, row in tqdm(academic_columns.iterrows(), total=academic_columns.shape[0], desc="Generating academic descriptions"):
    descriptions = generate_descriptions(row, academic_headers, academic_prompt, client)

    # Assigning values to new columns
    full_responses.at[index, 'academic_description_v1'] = descriptions[0]
    full_responses.at[index, 'academic_description_v2'] = descriptions[1]
    full_responses.at[index, 'academic_description_v3'] = descriptions[2]

print("Finished generating descriptions for academic information.")
time.sleep(10)

# Initialize new columns for professional experience information
full_responses['professional_description_v1'] = ""
full_responses['professional_description_v2'] = ""
full_responses['professional_description_v3'] = ""

# Headers for professional columns
professional_headers = professional_columns.columns.tolist()

# Prompts for professional experience descriptions
professional_prompt = """
Has the professional always worked at the same company, or have they changed employers?
Detail the professional's first job, including the year it started.
If the professional has changed companies, indicate whether they stayed within the same industry.
Describe the current role of the professional and specify if they hold a managerial or leadership position.
"""

# Iterate through each row and generate professional descriptions
for index, row in tqdm(professional_columns.iterrows(), total=professional_columns.shape[0], desc="Generating professional descriptions"):
    descriptions = generate_descriptions(row, professional_headers, professional_prompt, client)

    # Assigning values to new columns
    full_responses.at[index, 'professional_description_v1'] = descriptions[0]
    full_responses.at[index, 'professional_description_v2'] = descriptions[1]
    full_responses.at[index, 'professional_description_v3'] = descriptions[2]

print("Finished generating descriptions for professional experience information.")
time.sleep(10)

# Initialize new columns for innovation information
full_responses['innovation_description_v1'] = ""
full_responses['innovation_description_v2'] = ""
full_responses['innovation_description_v3'] = ""

# Headers for innovation columns
innovation_headers = innovation_columns.columns.tolist()

# Prompts for innovation descriptions
innovation_prompt = """
Has the professional been responsible for any innovations? If so, describe these innovations.
List the types of innovations the professional has contributed to, focusing on the nature of these contributions.
What is the total number of innovations the professional has been involved in?
Identify the type of innovation to which the professional has made the most significant contributions.
"""

# Iterate through each row and generate professional descriptions
for index, row in tqdm(innovation_columns.iterrows(), total=innovation_columns.shape[0], desc="Generating innovation descriptions"):
    descriptions = generate_descriptions(row, innovation_headers, innovation_prompt, client)

    # Assigning values to new columns
    full_responses.at[index, 'innovation_description_v1'] = descriptions[0]
    full_responses.at[index, 'innovation_description_v2'] = descriptions[1]
    full_responses.at[index, 'innovation_description_v3'] = descriptions[2]

print("Finished generating descriptions for innovation information.")
time.sleep(10)

# Initialize new columns for entrepreneurship information
full_responses['entrepreneurship_description_v1'] = ""
full_responses['entrepreneurship_description_v2'] = ""
full_responses['entrepreneurship_description_v3'] = ""

# Headers for entrepreneurship columns
entrepreneurship_headers = entrepreneurship_columns.columns.tolist()

# Prompts for entrepreneurship descriptions
entrepreneurship_prompt = """
Has the professional been an entrepreneur, investor, early employee, or board member in an entrepreneurial firm? Indicate if this was a one-time involvement or if they have been involved in multiple entrepreneurial ventures.
Identify the industries in which the professional has been involved as part of an entrepreneurial firm.
Specify the year of the professional's first entrepreneurial activity and whether their role was as a founder, early employee, investor, or board member. Include the industry and country of incorporation for the first venture.
Has the professional been involved in an exit or IPO process for any of the entrepreneurial firms they have been part of? Provide details.
"""

# Iterate through each row and generate entrepreneurship descriptions
for index, row in tqdm(entrepreneurship_columns.iterrows(), total=entrepreneurship_columns.shape[0], desc="Generating entrepreneurship descriptions"):
    descriptions = generate_descriptions(row, entrepreneurship_headers, entrepreneurship_prompt, client)

    # Assigning values to new columns
    full_responses.at[index, 'entrepreneurship_description_v1'] = descriptions[0]
    full_responses.at[index, 'entrepreneurship_description_v2'] = descriptions[1]
    full_responses.at[index, 'entrepreneurship_description_v3'] = descriptions[2]

print("Finished generating descriptions for entrepreneurship information.")
time.sleep(10)

# Initialize new columns for motivation information
full_responses['main_motivation_v1'] = ""
full_responses['main_motivation_v2'] = ""
full_responses['main_motivation_v3'] = ""

# Headers for motivation columns
motivation_headers = motivation_columns.columns.tolist()

# Prompts for motivation descriptions
motivation_prompt = """
The professional rated their main motivations for being involved in an entrepreneurial firm on a scale from 0 to 5. Summarize these motivations, highlighting the most significant reasons and any additional insights provided by the professional.
"""

# Iterate through each row and generate motivation descriptions
for index, row in tqdm(motivation_columns.iterrows(), total=motivation_columns.shape[0], desc="Generating motivation descriptions"):
    descriptions = generate_descriptions(row, motivation_headers, motivation_prompt, client)

    # Assigning values to new columns
    full_responses.at[index, 'main_motivation_v1'] = descriptions[0]
    full_responses.at[index, 'main_motivation_v2'] = descriptions[1]
    full_responses.at[index, 'main_motivation_v3'] = descriptions[2]

print("Finished generating descriptions for motivation information.")
time.sleep(10)

# Initialize new columns for motivation information
full_responses['cv_v1'] = ""
full_responses['cv_v2'] = ""
full_responses['cv_v3'] = ""

# Headers for CV columns
cv_headers = cv_columns.columns.tolist()

# Prompts for CV descriptions
cv_prompt = """
Based on the CV provided, create a short bio for the professional, detailing their academic background and professional trajectory. Include descriptions of their soft skills, languages spoken, certifications obtained, and any other relevant information that contributes to a comprehensive profile.
"""

# Iterate through each row and generate motivation descriptions
for index, row in tqdm(cv_columns.iterrows(), total=cv_columns.shape[0], desc="Generating CV descriptions"):
    descriptions = generate_descriptions(row, cv_headers, cv_prompt, client)

    # Assigning values to new columns
    full_responses.at[index, 'cv_v1'] = descriptions[0]
    full_responses.at[index, 'cv_v2'] = descriptions[1]
    full_responses.at[index, 'cv_v3'] = descriptions[2]

print("Finished generating descriptions for CV information.")

# Save the updated dataframe back to a CSV file
output_csv_path = output_dir / "full_responses_with_descriptions.csv"
full_responses.to_csv(output_csv_path, index=False)
print(f"Updated dataframe saved to {output_csv_path}")


## Instruction combinations (source cell 6)

Combines three CV variants, nine academic/professional/motivation variants, three answer variants, and two fixed tasks: 162 rows per respondent. The source's manual message-string construction is retained; it does not guarantee valid JSON when generated text contains quotes or other special characters. This is an archival intermediate, not a validated drop-in training dataset.


In [ ]:
# Load the dataset
file_path = output_dir / "full_responses_with_descriptions.csv"
full_responses = pd.read_csv(file_path)

# Columns containing generated descriptions
cv_columns = ['cv_v1', 'cv_v2', 'cv_v3']
context_columns = [
    'academic_description_v1', 'academic_description_v2', 'academic_description_v3',
    'professional_description_v1', 'professional_description_v2', 'professional_description_v3',
    'main_motivation_v1', 'main_motivation_v2', 'main_motivation_v3'
]
innovation_columns = ['innovation_description_v1', 'innovation_description_v2', 'innovation_description_v3']
entrepreneurship_columns = [
    'entrepreneurship_description_v1', 'entrepreneurship_description_v2', 'entrepreneurship_description_v3'
]

# Initialize a list to store the new instruction dataset
instruction_data = []

# Function to create prompt, question, and answer columns
def create_instruction(row, cv_col, context_col, innovation_col, question_text):
    prompt = row[cv_col]
    question = f"{row[context_col]} {question_text}"
    answer = row[innovation_col]
    return prompt, question, answer

# Iterate over each row to create instructions for innovation activities
for index, row in full_responses.iterrows():
    for cv_col, context_col, innovation_col in product(cv_columns, context_columns, innovation_columns):
        prompt, question, answer = create_instruction(row, cv_col, context_col, innovation_col, "Did this professional engage in any innovative activities? Was he or she responsible for any innovation?")
        # Create the new "chosen" column
        chosen_text = f'[ {{ "content": "{prompt}", "role": "user" }}, {{ "content": "What do you know about this profile?", "role": "assistant" }}, {{ "content": "{question}", "role": "user" }}, {{ "content": "{answer}", "role": "assistant" }} ]'
        instruction_data.append({'prompt': prompt, 'question': question, 'answer': answer, 'chosen': chosen_text})

# Iterate over each row to create instructions for entrepreneurial activities
for index, row in full_responses.iterrows():
    for cv_col, context_col, entrepreneurship_col in product(cv_columns, context_columns, entrepreneurship_columns):
        prompt, question, answer = create_instruction(row, cv_col, context_col, entrepreneurship_col, "Did this professional engage in any entrepreneurial activities? Was he or she an entrepreneur?")
        # Create the new "chosen" column with the same logic
        chosen_text = f'[ {{ "content": "{prompt}", "role": "user" }}, {{ "content": "What do you know about this profile?", "role": "assistant" }}, {{ "content": "{question}", "role": "user" }}, {{ "content": "{answer}", "role": "assistant" }} ]'
        instruction_data.append({'prompt': prompt, 'question': question, 'answer': answer, 'chosen': chosen_text})

# Convert the list to a DataFrame
instruction_data_df = pd.DataFrame(instruction_data)

# Save the instruction dataset
output_csv_path = output_dir / "instruction_dataset.csv"
instruction_data_df.to_csv(output_csv_path, index=False)
print(f"Instruction dataset saved to {output_csv_path}")


## Contrasting answers (source cell 7)

The active path calls `gpt-3.5-turbo` once for each preferred-answer record. Its historical resolved snapshot is unknown. The source's `replace_yes_with_no` helper is retained but is not called; it implements only Yes-to-No substitution. Do not describe that unused helper as the contrast-generation method.


In [ ]:
# Load the dataset
file_path = output_dir / "instruction_dataset.csv"
orpo_data = pd.read_csv(file_path)
client = preparation_client()

# Convert the list to a DataFrame
orpo_data = pd.DataFrame(orpo_data)

# Function to generate wrong answers using OpenAI API
def generate_wrong_answer(correct_answer, client):
    base_prompt = "Consider the right answer that I am pasting below, please provide a wrong answer without any introduction to the wrong answer itself. The wrong answer should be opposite, if the answer is positive you need to provide answer as negative. Answer: "
    prompt = base_prompt + correct_answer
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are a helpful research assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.8,
        max_tokens=500
    )
    wrong_answer = response.choices[0].message.content
    return wrong_answer

# Identify and replace "Yes" or "yes" with "No" or "no" in "answer" and copy to "wrong_answer"
def replace_yes_with_no(answer):
    if "Yes" in answer or "yes" in answer:
        return answer.replace("Yes", "No").replace("yes", "no")
    return ""

# Add a progress bar to the generation of wrong answers
tqdm.pandas()
orpo_data['wrong_answer'] = orpo_data['answer'].progress_apply(lambda x: generate_wrong_answer(x, client))

# Add a progress bar to the creation of the rejected column
orpo_data['rejected'] = orpo_data.progress_apply(
    lambda row: f'[ {{ "content": "{row["prompt"]}", "role": "user" }}, {{ "content": "What do you know about this profile?", "role": "assistant" }}, {{ "content": "{row["question"]}", "role": "user" }}, {{ "content": "{row["wrong_answer"]}", "role": "assistant" }} ]',
    axis=1
)

# Save the updated instruction dataset
output_csv_path = output_dir / "orpo_data.csv"
orpo_data.to_csv(output_csv_path, index=False)
print(f"ORPO instruction dataset saved to {output_csv_path}")


## Cleanup alternative A: full names (source cell 8)

This path reads `orpo_data.csv` and writes `orpo_training.csv`. The next section independently rereads `orpo_data.csv`; it is an alternative cleanup path, not an additional pass over this output. Both appear after API generation in the written source and do not prove pre-call filtering.


In [ ]:
# Load your dataset
file_path = output_dir / "orpo_data.csv"
orpo_data = pd.read_csv(file_path)

# Load your dataset
file_path = survey_path
survey_data = pd.read_csv(file_path)

# Extract all the names in the "Full names" column and build a dictionary
full_names = survey_data['Full name'].dropna().unique().tolist()
name_dict = {name: "<PERSON>" for name in full_names}

# Function to replace full names with "<PERSON>"
def replace_names(text, name_dict):
    if pd.isna(text):
        return text
    for name in name_dict:
        text = re.sub(rf"\b{name}\b", "<PERSON>", text)
    return text

# Check all cells in orpo_data and replace full names
for column in tqdm(orpo_data.columns, desc="Processing columns"):
    orpo_data[column] = orpo_data[column].apply(lambda x: replace_names(x, name_dict))

# Save the updated instruction dataset
output_csv_path = output_dir / "orpo_training.csv"
orpo_data.to_csv(output_csv_path, index=False)
print(f"ORPO training dataset saved to {output_csv_path}")

## Cleanup alternative B: name parts and placeholders (source cell 9)

This path independently reads the original `orpo_data.csv` and writes `orpo_training_updated.csv`. Regex, accent-normalization order, counters, placeholder spelling, and message-string handling are retained. The phone placeholder targeted here is `<PHONE NUMBER>`, not Presidio's `<PHONE_NUMBER>`. Removing placeholders is not equivalent to detecting raw identifiers. No upstream 90/10 split or Hub publication is included.


In [ ]:
import pandas as pd
import re
import unicodedata
from tqdm import tqdm

# Load your datasets
orpo_data_path = output_dir / "orpo_data.csv"
survey_data_path = survey_path
orpo_data = pd.read_csv(orpo_data_path)
survey_data = pd.read_csv(survey_data_path)

# Extract all unique words in the "Full names" column and build a dictionary
full_names = survey_data['Full name'].dropna().unique().tolist()
unique_words = set(word for name in full_names for word in name.split())
name_dict = {word: "<PERSON>" for word in unique_words}

# Counters to track the number of changes made
person_replace_counter = 0
phone_remove_counter = 0
email_remove_counter = 0
url_remove_counter = 0

# Function to remove accents from words
def remove_accents(text):
    nfkd_form = unicodedata.normalize('NFKD', text)
    return "".join([c for c in nfkd_form if not unicodedata.combining(c)])

# Function to replace individual words from full names with "<PERSON>"
def replace_names(text, name_dict):
    global person_replace_counter
    if pd.isna(text):
        return text
    original_text = text
    for word in name_dict:
        text = re.sub(rf"\b{word}\b", "<PERSON>", text, flags=re.IGNORECASE)
    if text != original_text:
        person_replace_counter += 1
    return text

# Function to clean and count replacements/removals
def clean_text(text):
    global phone_remove_counter, email_remove_counter, url_remove_counter
    if pd.isna(text):
        return text
    text = remove_accents(text)
    text = text.replace("[NAME]", "person")
    text = text.replace("<PERSON>", "person")
    text = text.replace("<NAME>", "person")
    text = re.sub(r"\b(person person)\b", "person", text, flags=re.IGNORECASE)
    phone_count = text.count("<PHONE NUMBER>")
    email_count = text.count("<EMAIL_ADDRESS>")
    url_count = text.count("<URL>")
    text = text.replace("<PHONE NUMBER>", "")
    text = text.replace("<EMAIL_ADDRESS>", "")
    text = text.replace("<URL>", "")
    phone_remove_counter += phone_count
    email_remove_counter += email_count
    url_remove_counter += url_count
    return text

# Check all cells in orpo_data and replace full names and clean text
for column in tqdm(orpo_data.columns, desc="Processing columns"):
    orpo_data[column] = orpo_data[column].apply(lambda x: replace_names(x, name_dict))
    orpo_data[column] = orpo_data[column].apply(clean_text)

print(f"Total 'person' replacements made: {person_replace_counter}")
print(f"Total '<PHONE NUMBER>' removals made: {phone_remove_counter}")
print(f"Total '<EMAIL_ADDRESS>' removals made: {email_remove_counter}")
print(f"Total '<URL>' removals made: {url_remove_counter}")

# Duplicate "prompt" column and rename it to "context"
orpo_data['context'] = orpo_data['prompt']

# Concatenate "prompt" and "question" with a newline in between and save them in "prompt"
orpo_data['prompt'] = orpo_data['prompt'] + '\n' + orpo_data['question']

# Function to clean JSON strings
def clean_json_string(json_str):
    if pd.isna(json_str):
        return json_str
    json_str = re.sub(r'[\x00-\x1F]+', '', json_str)
    return json_str

# Clean the JSON strings in the "chosen" and "rejected" columns
orpo_data['chosen'] = orpo_data['chosen'].apply(clean_json_string)
orpo_data['rejected'] = orpo_data['rejected'].apply(clean_json_string)

# Reorder columns and retain only the required columns
orpo_data = orpo_data[['context', 'question', 'prompt', 'chosen', 'rejected']]

# Save the updated dataset
output_csv_path = output_dir / "orpo_training_updated.csv"
orpo_data.to_csv(output_csv_path, index=False)

print(f"Updated dataset saved to {output_csv_path}")